In [6]:
import pandas as pd
import os

os.chdir("/Users/yegberink/Documents/geothermal_text_mining")

paragraphs =pd.read_csv("output/italian/text/paragraph_locations_ollama.csv")


In [7]:
print(paragraphs["llm_location"].value_counts())
print(paragraphs.columns)

llm_location
Italy                177
Toscana              144
Larderello            58
Ferrara               57
Piancastagnaio        52
                    ... 
Chiusdino              1
Terra                  1
Piazza dell'acqua      1
Bergamo                1
Kenya                  1
Name: count, Length: 241, dtype: int64
Index(['source_file', 'source_path', 'source_relative_path', 'source_folder',
       'title', 'newspaper', 'date', 'section', 'word_count', 'load_date',
       'body', 'date_translated', 'date_parsed', 'geo_hits', 'body_hash',
       'source', 'document_title', 'publish_date', 'region_name',
       'paragraph_id', 'paragraph_text', 'paragraph_word_count',
       'paragraph_sentence_count', 'split_reason', 'layout_block_id',
       'layout_block_word_count', 'original_layout_block_text',
       'semantic_similarity_before', 'semantic_similarity_after', 'uid',
       'llm_is_geothermal', 'llm_geo_confidence', 'llm_geo_evidence_short',
       'llm_status', 'llm_error'

In [8]:
sentences =pd.read_csv("output/dutch/text/sentences_with_categories_admin.csv")

print(sentences.columns)


Index(['source_file', 'source_path', 'title', 'newspaper', 'date', 'section',
       'word_count', 'load_date', 'body', 'date_translated', 'date_parsed',
       'geo_hits', 'body_hash', 'source', 'document_title', 'publish_date',
       'region_name', 'paragraph_id', 'paragraph_text', 'llm_is_geothermal',
       'llm_geo_confidence', 'llm_geo_evidence_short', 'llm_status',
       'llm_error', 'llm_location', 'llm_granularity', 'llm_confidence',
       'llm_reasoning_short', 'uid', 'paragraph_uid', 'sentence_id',
       'sentence_count_in_paragraph', 'sentence_text', 'sentence_word_count',
       'sentence_char_count', 'sentence_uid', 'n_categories',
       'matched_categories_str', 'matched_keywords_str', 'sentiment',
       'sentiment_norm', 'sentiment_confidence', 'sentiment_evidence_short',
       'sentiment_status', 'sentiment_error', '_gran', '_loc_first',
       '_loc_norm', '_join_key', 'geo_level', 'geo_name_matched', 'geo_source',
       'geo_match_type', 'geo_lat', 'geo_lon',

In [13]:
#explode sentences into separate rows so each row has a single category
#categories are split by ";"

sentences =pd.read_csv("output/dutch/text/sentences_with_categories_admin.csv")

sentences["matched_keywords_str"] = sentences["matched_keywords_str"].str.split(";")
sentences["matched_categories_str"] = sentences["matched_categories_str"].str.split(";")
sentences_exploded = sentences.explode("matched_categories_str")

#count the number of sentences per category
category_counts = sentences_exploded["matched_categories_str"].value_counts()
print(category_counts)

# print top 10 keywords per category
top_keywords = sentences_exploded.groupby("matched_categories_str")["matched_keywords_str"].apply(lambda x: pd.Series(x).explode().value_counts().head(10))
print(top_keywords)


matched_categories_str
Costs                      430
Risks                      341
Sustainability             283
Governance                 268
Knowledge availability     162
Support                     81
Comparing locations         38
Technological readiness     25
Name: count, dtype: int64
matched_categories_str                   
Comparing locations      duitsland           17
                         ijsland             11
                         barendrecht          6
                         draagvlak            3
                         boxtel               3
                                             ..
Technological readiness  pilotproject         2
                         opschaling           2
                         barendrecht          1
                         draagvlak            1
                         exploratieboring     1
Name: matched_keywords_str, Length: 80, dtype: int64


In [14]:
from IPython.display import display

review = sentences.copy()
review["sentence_id"] = review.index

def split_semicolon_values(value):
    if isinstance(value, list):
        values = value
    elif pd.isna(value):
        values = []
    elif isinstance(value, str):
        values = value.split(";")
    else:
        values = str(value).split(";")
    return [v.strip() for v in values if str(v).strip()]

for col in ["matched_categories_str", "matched_keywords_str"]:
    review[col] = review[col].apply(split_semicolon_values)

lengths_match = review["matched_categories_str"].str.len() == review["matched_keywords_str"].str.len()
mismatch_count = (~lengths_match).sum()
print(f"Rows where category and keyword list lengths differ: {mismatch_count}")
if mismatch_count:
    display(review.loc[~lengths_match, ["matched_categories_str", "matched_keywords_str"]].head(10))

# Keep the current notebook logic: each category gets the full keyword list from its sentence.
# This avoids dropping information in rows where category and keyword counts do not align 1:1.
category_rows = review.explode("matched_categories_str").rename(columns={"matched_categories_str": "category"})
category_rows = category_rows.dropna(subset=["category"])
category_rows = category_rows[category_rows["category"] != ""]

pairs = category_rows.explode("matched_keywords_str").rename(columns={"matched_keywords_str": "keyword"})
pairs = pairs.dropna(subset=["keyword"])
pairs = pairs[pairs["keyword"] != ""]

category_sentence_counts = (
    category_rows[["sentence_id", "category"]]
    .drop_duplicates()
    .groupby("category")
    .size()
    .rename("n_sentences")
)

top_keywords_tidy = (
    pairs.groupby(["category", "keyword"])
    .size()
    .rename("count")
    .reset_index()
    .sort_values(["category", "count", "keyword"], ascending=[True, False, True])
)
top_keywords_tidy["rank"] = top_keywords_tidy.groupby("category").cumcount() + 1
top_keywords_tidy = top_keywords_tidy[top_keywords_tidy["rank"] <= 10]
top_keywords_tidy = top_keywords_tidy.merge(category_sentence_counts, on="category", how="left")
top_keywords_tidy["keyword_share"] = top_keywords_tidy["count"] / top_keywords_tidy["n_sentences"]

# Compact review table: one row per category, top keywords shown as keyword (count)
review_table = (
    top_keywords_tidy.assign(label=lambda df: df["keyword"] + " (" + df["count"].astype(str) + ")")
    .pivot(index="category", columns="rank", values="label")
    .rename(columns=lambda rank: f"top_{rank}")
)
review_table = category_sentence_counts.to_frame().join(review_table, how="left").reset_index()
review_table = review_table.rename(columns={"index": "category"}).fillna("")

display(
    review_table.style
    .set_caption("Top keywords per category")
    .background_gradient(subset=["n_sentences"], cmap="YlGnBu")
    .hide(axis="index")
)

# Save outputs for manual checking outside the notebook
review_table.to_csv("output/dutch/figures/top_keywords_review.csv", index=False)
review_table.to_html("output/dutch/figures/top_keywords_review.html", index=False)

# Optional visual check: faceted bars, one panel per category
try:
    import plotly.express as px

    chart_df = top_keywords_tidy.sort_values(["category", "count", "keyword"], ascending=[True, True, True])
    fig = px.bar(
        chart_df,
        x="count",
        y="keyword",
        facet_col="category",
        facet_col_wrap=3,
        orientation="h",
        text="count",
        height=950,
        title="Top 10 keywords per category",
    )
    fig.for_each_annotation(lambda ann: ann.update(text=ann.text.split("=")[-1]))
    fig.update_yaxes(matches=None)
    fig.update_layout(showlegend=False)

    chart_path = "output/dutch/figures/top_keywords_review_plotly.html"
    fig.write_html(chart_path)
    print(f"Saved Plotly chart to {chart_path}")

    try:
        fig.show()
    except ValueError as exc:
        print(f"Skipping inline Plotly display: {exc}")
except ImportError:
    print("Install plotly if you want the faceted chart: pip install plotly")

top_keywords_tidy.head(20)


Rows where category and keyword list lengths differ: 49


,matched_categories_str,matched_keywords_str
20,[Risks],"[risico, bevingen]"
59,"[Knowledge availability, Sustainability]","[kennis, duurzame warmte, duurzame energiebron]"
105,[Risks],"[risico, seismische activiteit]"
128,[Governance],"[vergunning, besluit]"
177,[Costs],"[investering, kosten]"
192,[Risks],"[probleem, bodemdaling]"
200,[Costs],"[operationele kosten, subsidie]"
219,"[Sustainability, Costs]","[duurzame warmte, operationele kosten, subsidie]"
306,[Comparing locations],"[duitsland, ijsland]"
359,[Risks],"[aardbeving, magnitude]"


category,n_sentences,top_1,top_2,top_3,top_4,top_5,top_6,top_7,top_8,top_9,top_10
Comparing locations,38,duitsland (17),ijsland (11),barendrecht (6),boxtel (3),draagvlak (3),zuid-korea (2),aardschok (1),duur (1),proeftuin (1),zwitserland (1)
Costs,430,kosten (133),subsidie (100),investering (36),financieel (29),duur (28),duurder (24),rendabel (22),financiering (16),budget (12),goedkoper (12)
Governance,268,vergunning (109),besluit (64),goedkeuring (18),regelgeving (17),beleid (12),informatieavond (10),besluitvorming (8),bestemmingsplan (7),debat (7),risico (7)
Knowledge availability,162,seismisch onderzoek (88),kennis (54),geologisch onderzoek (8),boorresultaat (7),doorlatendheid (3),risico (3),duurzame warmte (2),bestemmingsplannen (1),duurzame energiebron (1),duurzame warmtebron (1)
Risks,341,risico (178),trillingen (61),probleem (38),bevingen (23),aardbeving (14),bodemdaling (13),onzekerheid (13),seismische activiteit (7),kosten (5),goedkeuring (3)
Support,81,draagvlak (22),steun (22),vertrouwen (19),weerstand (10),actiegroep (4),duurder (4),informatieavond (4),barendrecht (3),besluit (2),onrust (2)
Sustainability,283,duurzame energie (83),duurzame warmte (68),duurzame energiebron (26),groene energie (23),klimaatdoelen (14),duurzame warmtebron (13),schone energie (12),duurzame bron (10),co2-neutraal (8),schone warmte (8)
Technological readiness,25,proeftuin (6),experiment (5),pilot (4),projectontwikkeling (3),opschaling (2),pilotproject (2),testfase (2),barendrecht (1),budget (1),draagvlak (1)


Saved Plotly chart to output/dutch/figures/top_keywords_review_plotly.html
Skipping inline Plotly display: Mime type rendering requires nbformat>=4.2.0 but it is not installed


,category,keyword,count,rank,n_sentences,keyword_share
0,Comparing locations,duitsland,17,1,38,0.447368
1,Comparing locations,ijsland,11,2,38,0.289474
2,Comparing locations,barendrecht,6,3,38,0.157895
3,Comparing locations,boxtel,3,4,38,0.078947
4,Comparing locations,draagvlak,3,5,38,0.078947
5,Comparing locations,zuid-korea,2,6,38,0.052632
6,Comparing locations,aardschok,1,7,38,0.026316
7,Comparing locations,duur,1,8,38,0.026316
8,Comparing locations,proeftuin,1,9,38,0.026316
9,Comparing locations,zwitserland,1,10,38,0.026316
